In [37]:
df = pd.read_csv(
    r"C:\Users\GIFTY\Projects\customer-analytics-portfolio\customer-lifetime-value-analysis\data\raw\online_retail_raw.csv",
    encoding="latin1"
)
df.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom


In [38]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']


In [39]:
df[['CustomerID', 'InvoiceDate', 'Revenue']].info()
df[['Revenue']].describe()
print("Total No of Customers :", df['CustomerID'].nunique())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   CustomerID   406829 non-null  float64
 1   InvoiceDate  541909 non-null  object 
 2   Revenue      541909 non-null  float64
dtypes: float64(2), object(1)
memory usage: 12.4+ MB
Total No of Customers : 4372


In [40]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
Revenue             0
dtype: int64

In [41]:
df = df.dropna(subset=['CustomerID'])
df = df[df['Revenue'] > 0]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


In [42]:
min_date = df['InvoiceDate'].min()
max_date = df['InvoiceDate'].max()

months = (max_date.year - min_date.year) * 12 + (max_date.month - min_date.month)

min_date, max_date, months


(Timestamp('2010-12-01 08:26:00'), Timestamp('2011-12-09 12:50:00'), 12)

> Transactions span from 2010-12-01 to 2011-12-09 (~12 months)

In [43]:
purchase_counts = (
    df.groupby('CustomerID')['InvoiceNo']
    .nunique()
)


In [44]:
purchase_counts.describe()


count    4338.000000
mean        4.272015
std         7.697998
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       209.000000
Name: InvoiceNo, dtype: float64

In [45]:
purchase_counts.value_counts().head(10)


InvoiceNo
1     1493
2      835
3      508
4      388
5      242
6      172
7      143
8       98
9       68
10      54
Name: count, dtype: int64

In [46]:
df_sorted = df.sort_values(['CustomerID', 'InvoiceDate'])

df_sorted['prev_date'] = df_sorted.groupby('CustomerID')['InvoiceDate'].shift(1)
df_sorted['inter_purchase_days'] = (
    df_sorted['InvoiceDate'] - df_sorted['prev_date']
).dt.days

df_sorted['inter_purchase_days'].describe()


count    393546.000000
mean          1.425361
std          12.297970
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         365.000000
Name: inter_purchase_days, dtype: float64

In [47]:
df[['CustomerID', 'InvoiceDate', 'Revenue']].info()


<class 'pandas.core.frame.DataFrame'>
Index: 397884 entries, 0 to 541908
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   CustomerID   397884 non-null  float64       
 1   InvoiceDate  397884 non-null  datetime64[ns]
 2   Revenue      397884 non-null  float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 12.1 MB
